# Python 101 - Solutions
## Chapter XIII

---

**For teaching assistants.** This notebook mirrors the exercises in
`../python101_13.ipynb` one for one. Most solutions end with an `assert`, so
running the whole notebook top to bottom is also a self-test: if it runs clean,
every solution still works.

There is usually more than one right answer - if a student's version passes the
same `assert`, it is correct.

In [ ]:
# Run from the chapter folder, so that `helpers`, `./data/...` and `./pics/...`
# resolve exactly the way they do in the lecture notebooks.
import os
import sys

if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

print('working directory:', os.getcwd())

> **Note.** The course now runs to 12 weeks and rarely reaches this chapter. Solutions here for completeness.

In [ ]:
%matplotlib inline
import collections
import itertools
import operator
from functools import partial, reduce

import matplotlib.pyplot as plt
import seaborn as sns

from helpers import FakeMapReduce

### Exercise: the first 10 binary digits with `map` and `lambda`

'Binary digit' here means the powers of two - `2**n` for n in 0..9.

In [ ]:
binary_digits = list(map(lambda x: 2 ** x, range(10)))
print(binary_digits)

assert binary_digits == [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]

### Exercise: `filter` with a lambda, then as a comprehension

Note `x % 2` is truthy for odd numbers - so this keeps the **odd** ones, despite the function being called `odd`. Worth reading carefully.

In [ ]:
with_lambda = list(filter(lambda x: x % 2, range(10)))
with_comprehension = [x for x in range(10) if x % 2]

print(with_lambda)
print(with_comprehension)

assert with_lambda == with_comprehension == [1, 3, 5, 7, 9]

### Exercise: 10! with `reduce`

`range(1, 11)` - starting from 0 would make the whole product 0, which is a mistake worth letting them make.

In [ ]:
factorial = reduce(lambda x, y: x * y, range(1, 11))
print(factorial)

assert factorial == 3628800
assert reduce(operator.mul, range(1, 11)) == factorial       # tidier
assert reduce(lambda x, y: x * y, range(11)) == 0            # the trap

### Exercise: merge a list of lists into one list

In [ ]:
nested = [range(i) for i in range(1, 6)]

merged = reduce(lambda x, y: list(x) + list(y), nested)
print(merged)

assert merged == [0, 0, 1, 0, 1, 2, 0, 1, 2, 3, 0, 1, 2, 3, 4]
# the comprehension version, for comparison
assert merged == [item for sub in nested for item in sub]
# and the itertools version
assert merged == list(itertools.chain.from_iterable(nested))

### The wordcount pipeline

One thing to demonstrate live: `map` and `filter` are **lazy**. Each of these objects can only be walked once, which is why the notebook calls `list(...)` after every step - and why running the cells out of order gives you empty results. That confuses everybody the first time.

In [ ]:
text = ("This is your last chance. "
        "After this, there is no turning back... "
        "You take the blue pill, the story ends. "
        "You wake up and belive... whatever you want to believe. "
        "You take the red pill... you stay in wonderland... "
        "and I show you just how deep the rabbit hole goes."
        "Remember...all I'm offering you is the truth : nothing more.")
unwanted_chars = '.:'
words2remove = []

char_filter = lambda x: x not in unwanted_chars


def wordcount(text):
    lower = map(str.lower, text.split())
    cleared = map(lambda word: ''.join(filter(char_filter, word)), lower)
    kept = filter(lambda word: word not in words2remove, cleared)
    pairs = map(lambda word: (word, 1), kept)

    def add(counts, pair):
        word, count = pair
        counts[word] += count
        return counts

    return reduce(add, pairs, collections.defaultdict(int))


counts = wordcount(text)
print(sorted(counts.items(), key=lambda pair: -pair[1])[:6])

assert counts['you'] == 7
assert counts['the'] == 5
# note 'this' and 'this,' collapse to one word only because the ',' is NOT in
# unwanted_chars - so they stay separate. Only '.' and ':' get stripped.
assert counts['this'] == 1 and counts['this,'] == 1

# laziness, demonstrated
once = map(str.lower, ['A', 'B'])
assert list(once) == ['a', 'b']
assert list(once) == []          # exhausted - this is the gotcha
print('\nsecond pass over the same map object is empty - that is laziness')

### The `FakeMapReduce` version

Same pipeline, spark-shaped. Now that `FakeMapReduce` keeps lists rather than lazy iterators, printing it actually shows the values.

In [ ]:
result = (FakeMapReduce(text.split())
          .map(lambda x: x.lower())
          .map(lambda x: ''.join(filter(char_filter, x)))
          .filter(lambda x: x not in words2remove)
          .map(lambda x: (x, 1))
          .reduceByKey(lambda x, y: x + y))

print(result)

assert result.collect()['you'] == 7
assert result.collect() == dict(counts)
print('\nsame answer as the reduce() version')

### Exercise: the sum of the even numbers 0-10, with `operator`

In [ ]:
evens = filter(lambda x: not x % 2, range(11))
total = reduce(operator.add, evens)
print(total)

assert total == 0 + 2 + 4 + 6 + 8 + 10 == 30
assert sum(range(0, 11, 2)) == 30        # ...and the way you would really write it

### Exercise: plot the function with different `a` values

$f(a, b) = a^2 - 2ab + 1$. `partial(f, a)` freezes the first argument, so each curve is one `map` over the same range.

In [ ]:
def f(a, b):
    return a ** 2 - 2 * a * b + 1


xs = range(10)

print(list(map(partial(f, 2), xs)))

plt.figure(figsize=(9, 5))
for a in range(10):
    plt.plot(list(xs), list(map(partial(f, a), xs)), label=f'a={a}')
plt.legend(ncol=2)
plt.title('$f(a, b) = a^2 - 2ab + 1$ for ten values of a');

assert list(map(partial(f, 2), range(3))) == [5, 1, -3]
assert list(map(partial(f, 0), xs)) == [1] * 10      # a=0 flattens it

### Exercise: the binary digits again, with `partial` instead of a lambda

`pow(base, exponent)` takes its arguments in that order, so `partial(pow, 2)` freezes the **base** - exactly what we want.

In [ ]:
binary_digits = list(map(partial(pow, 2), range(10)))
print(binary_digits)

assert binary_digits == [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
assert binary_digits == list(map(lambda x: 2 ** x, range(10)))

### Exercise: the instruction-list calculator

Each entry is `(function, numbers)`, so `reduce` applies the function across the numbers. Two traps in the data: `operator.add` over `range(10)` includes 0 (fine), but `operator.truediv` over `range(10)` starts with `0 / 1` and then divides by zero - so it needs handling.

In [ ]:
data = [(operator.add, range(10)),
        (operator.mul, range(1, 4)),
        (operator.truediv, range(10)),
        (lambda x, y: x ** y, range(2, 5))]


def run(instruction):
    function, numbers = instruction
    try:
        return reduce(function, numbers)
    except ZeroDivisionError:
        return None


results = list(map(run, data))
for (function, numbers), result in zip(data, results):
    name = getattr(function, '__name__', 'lambda')
    print(f'  {name:9s} over {list(numbers)} -> {result}')

assert results[0] == 45                 # 0+1+...+9
assert results[1] == 6                  # 1*2*3
# The trap is the other way round from what you would expect: reduce starts
# with 0 / 1 == 0.0, and from then on 0 is always the *numerator*, so there is
# never a division by zero - the answer is just 0.0. Start the range at 1 and
# you get a very small number instead.
assert results[2] == 0.0
assert reduce(operator.truediv, range(1, 10)) < 1e-5

# reduce is left-associative, so this is (2**3)**4 == 4096, not 2**(3**4)
assert results[3] == 4096
assert results[3] != 2 ** 3 ** 4
print(f'\n(2**3)**4 = {results[3]}, but 2**(3**4) = {2 ** 3 ** 4}')

# and with starmap, as the notebook suggests
assert list(itertools.starmap(operator.add, [(3, 4), (5, 6), (9, 10)])) == [7, 11, 19]

### Cool library of the week: ripgrep

`!rg -t py random` shells out to ripgrep. If it is not installed, `conda install -c conda-forge ripgrep`. Nothing to solve.